# 01 – Primeiros Passos com `pyglenn`

**`pyglenn`** é um calculador de propriedades termoquímicas leve e sem
dependências externas. Ele reconstrói três propriedades molares no estado
padrão como funções analíticas da temperatura,

$$C_p^\circ(T), \qquad H^\circ(T), \qquad S^\circ(T),$$

a partir de **coeficientes polinomiais da NASA** armazenados em um banco de
dados **SQLite** empacotado. O banco de dados acompanha o pacote e contém
aproximadamente **2030 espécies químicas** (gases e fases condensadas)
distribuídas em **3772 intervalos de temperatura**, derivadas do conjunto de
dados `thermo.inp` do NASA Glenn.

Este primeiro notebook cobre os fundamentos:

1. Conexão com o banco de dados empacotado
2. Inspeção do conteúdo do banco de dados
3. Busca por espécies
4. Cálculo de $C_p^\circ$, $H^\circ$ e $S^\circ$ em uma dada temperatura
5. Compreensão dos valores retornados
6. Diferenças de entalpia e tabelas de propriedades
7. Tratamento elegante de erros

> **Autor do pacote:** Dr. Reginaldo G. Leão Jr. — GESESC / IFMG.

## Conteúdo

1. [Conectando ao banco de dados](#1-conectando-ao-banco-de-dados)
2. [Inspecionando o banco de dados](#2-inspecionando-o-banco-de-dados)
3. [Buscando espécies](#3-buscando-espécies)
4. [Calculando propriedades em uma temperatura](#4-calculando-propriedades-em-uma-temperatura)
5. [Diferenças de entalpia (calor sensível)](#5-diferenças-de-entalpia-calor-sensível)
6. [Tabelas de propriedades para várias temperaturas](#6-tabelas-de-propriedades-para-várias-temperaturas)
7. [Tratamento de erros](#7-tratamento-de-erros)

## Instalação

`pyglenn` requer apenas Python ≥ 3.9 (SQLite3 está na biblioteca padrão).
Instale a partir do PyPI, conda-forge ou do código-fonte:

```bash
pip install pyglenn
# ou via conda:
conda install conda-forge::pyglenn numpy pandas matplotlib scipy
# ou, a partir de um repositório local:
pip install .
```

Os comandos abaixo assumem que o pacote já pode ser importado.

In [1]:
from pyglenn import ThermochemicalCalculator, R

print("Universal gas constant R =", R, "J/(mol.K)")


Universal gas constant R = 8.314462618 J/(mol.K)


## 1. Conectando ao banco de dados

`ThermochemicalCalculator` é o ponto de entrada de alto nível. Como ele gerencia
uma conexão SQLite, o padrão recomendado é o **gerenciador de contexto**, que
abre a conexão na entrada e a fecha na saída — mesmo que uma exceção seja
lançada:

```python
with ThermochemicalCalculator() as calc:
    ...  # calc está conectado aqui
# conexão automaticamente fechada aqui
```

Nenhum caminho é necessário: o construtor usa por padrão o arquivo `thermo.db`
empacotado dentro do pacote.

In [2]:
with ThermochemicalCalculator() as calc:
    print("Conectado? ", calc.connected)
print("Conectado após o bloco?", calc.connected)

Conectado?  True
Conectado após o bloco? False


### Gerenciamento manual da conexão

Se você preferir controle explícito (por exemplo, dentro de um serviço de longa
duração), pode chamar `connect()` e `close()` manualmente. `connect()` retorna
`True` em caso de sucesso.

In [3]:
calc = ThermochemicalCalculator()
ok = calc.connect()
print("connect() ->", ok)
print("calc.connected ->", calc.connected)
calc.close()
print("após close ->", calc.connected)

connect() -> True
calc.connected -> True
após close -> False


## 2. Inspecionando o banco de dados

O objeto de consulta subjacente é exposto como `calc.db` (uma instância de
`ThermoDBQuery`). Seu método `get_statistics()` fornece uma visão geral rápida
do conjunto de dados.

In [4]:
with ThermochemicalCalculator() as calc:
    stats = calc.db.get_statistics()

for key, value in stats.items():
    print(f"{key:22s}: {value}")

total_species         : 2030
total_intervals       : 3772
total_coeff_sets      : 3772
species_by_phase      : {'condensed': 766, 'gas': 1264}
avg_molecular_weight  : 3143009517.295669


Observe que `species_by_phase` distingue espécies no estado **gasoso** (gas)
das espécies **condensadas** (condensed — líquido/sólido). O banco de dados
mistura ambos, portanto, ao buscar um combustível ou um produto, você pode
encontrar várias entradas — uma fase gasosa, uma fase líquida e, às vezes,
múltiplas formas cristalinas.

## 3. Buscando espécies

`get_available_species(pattern)` retorna uma lista de dicionários. Com um
`pattern` não vazio, ele faz uma correspondência **parcial** tanto no nome
quanto na fórmula química (limitado a 20 resultados); com um padrão vazio, ele
pagina por todo o catálogo.

In [5]:
with ThermochemicalCalculator() as calc:
    matches = calc.get_available_species("H2O")

print(f"{len(matches)} correspondências para 'H2O':\n")
for s in matches:
    print(f"  id={s['id']:5d}  {s['name']:18s}  {s['phase']:9s}  M={s['molecular_weight']:.4f} g/mol")

8 correspondências para 'H2O':

  id=  672  H2O                 gas        M=18.0153 g/mol
  id=  293  CH2OH               gas        M=31.0339 g/mol
  id=  294  CH2OH+              gas        M=31.0334 g/mol
  id= 1510  H2O(L)              condensed  M=18.0153 g/mol
  id= 1509  H2O(cr)             condensed  M=18.0153 g/mol
  id=  673  H2O+                gas        M=18.0147 g/mol
  id=  674  H2O2                gas        M=34.0147 g/mol
  id=  847  NH2OH               gas        M=33.0299 g/mol


Como a busca é por substring, a consulta acima retorna `CH2OH`, `H2O2`,
`NH2OH`, ... além da própria água. Quando você precisa de uma espécie
específica, passe `exact_match=True` para uma busca exata sem distinção
de maiúsculas/minúsculas:

In [6]:
with ThermochemicalCalculator() as calc:
    for name in ["O2", "N2", "CO2", "H2O", "CH4", "Ar"]:
        sid = calc.get_available_species(name, exact_match=True)[0]["id"]
        print(f"{name:5s} -> id no banco de dados {sid}")

O2    -> id no banco de dados 931
N2    -> id no banco de dados 861
CO2   -> id no banco de dados 316
H2O   -> id no banco de dados 672
CH4   -> id no banco de dados 296
Ar    -> id no banco de dados 74


### Navegando por todo o catálogo

Passar uma string vazia pagina por todas as ~2030 espécies. Aqui apenas as
contamos e visualizamos as primeiras.

In [7]:
with ThermochemicalCalculator() as calc:
    everything = calc.get_available_species("")

print(f"Total de espécies no catálogo: {len(everything)}\n")
print("Primeiras 8 (em ordem alfabética):")
for s in everything[:8]:
    print(f"  {s['name']:20s}  {s['phase']}")

Total de espécies no catálogo: 2030

Primeiras 8 (em ordem alfabética):
  (CH3COOH)2            gas
  (HCOOH)2              gas
  (WO3)2                gas
  (WO3)3                gas
  (WO3)4                gas
  (WO3)5                gas
  -1.302004763D+06      condensed
  -7.310769440D+05      condensed


## 4. Calculando propriedades em uma temperatura

`calculate_properties(species_id, temperature_K)` é o método central. Ele
retorna um dicionário de propriedades no estado padrão avaliadas na temperatura
solicitada. Vamos avaliar o oxigênio molecular a 298,15 K e a 1000 K.

In [8]:
with ThermochemicalCalculator() as calc:
    o2 = calc.get_available_species("O2", exact_match=True)[0]["id"]
    for T in (298.15, 1000.0):
        props = calc.calculate_properties(o2, T)
        print(f"--- O2 a {T} K ---")
        for k, v in props.items():
            print(f"    {k:14s}: {v}")
        print()

--- O2 a 298.15 K ---
    temperature   : 298.15
    cp            : 29.37818595837342
    h_relative    : -1.2807210325893945e-05
    s             : 205.14829827953108
    temp_interval : [200.0, 1000.0]
    species_name  : O2
    phase         : gas

--- O2 a 1000.0 K ---
    temperature   : 1000.0
    cp            : 34.882346223554805
    h_relative    : 22707.08129523519
    s             : 243.58592574388805
    temp_interval : [200.0, 1000.0]
    species_name  : O2
    phase         : gas



### O que esses números significam?

| Chave | Significado | Unidades |
|-----|---------|-------|
| `temperature`   | A temperatura solicitada | K |
| `cp`            | Capacidade calorífica no estado padrão $C_p^\circ(T)$ | J/(mol·K) |
| `s`             | Entropia absoluta no estado padrão (Terceira Lei) $S^\circ(T)$ | J/(mol·K) |
| `h_relative`    | Entalpia molar padronizada $H^\circ(T)$ na escala NASA | J/mol |
| `temp_interval` | `[T_min, T_max]` do trecho polinomial utilizado | K |
| `species_name`  | Nome da espécie resolvida | — |
| `phase`         | `gas` (gás) ou `condensed` (condensado) | — |

Um ponto crucial sobre **`h_relative`**: os polinômios da NASA ajustam a
entalpia *padronizada*, que **já inclui a entalpia de formação**. Como
consequência:

* para uma espécie em seu estado de referência elementar (ex.: O₂, N₂,
  grafite), $H^\circ(298{,}15\,\text{K}) \approx 0$;
* para um composto, $H^\circ(298{,}15\,\text{K})$ é igual à sua **entalpia de
  formação** $\Delta_f H^\circ$.

É por isso que o `h_relative` do oxigênio acima é essencialmente zero a
298,15 K. Exploramos essa propriedade em notebooks posteriores para calcular
entalpias de reação diretamente. Verificação rápida para o vapor d'água:

In [9]:
with ThermochemicalCalculator() as calc:
    h2o = calc.get_available_species("H2O", exact_match=True)[0]["id"]
    hf = calc.calculate_properties(h2o, 298.15)["h_relative"]
print(f"Entalpia padronizada do H2O(g) a 298,15 K = {hf/1000:8.2f} kJ/mol")
print("Entalpia de formação da literatura para H2O(g) = −241,83 kJ/mol")

Entalpia padronizada do H2O(g) a 298,15 K =  -241.82 kJ/mol
Entalpia de formação da literatura para H2O(g) = −241,83 kJ/mol


## 5. Diferenças de entalpia (calor sensível)

A variação de entalpia necessária para aquecer um mol de uma substância de
$T_1$ até $T_2$ (sua entalpia *sensível*) é $H^\circ(T_2) - H^\circ(T_1)$.
`calculate_enthalpy_change` calcula isso diretamente.

In [10]:
with ThermochemicalCalculator() as calc:
    n2 = calc.get_available_species("N2", exact_match=True)[0]["id"]
    dH = calc.calculate_enthalpy_change(n2, 300.0, 1200.0)
print(f"Aquecer 1 mol de N2 de 300 K a 1200 K requer {dH/1000:.2f} kJ")

Aquecer 1 mol de N2 de 300 K a 1200 K requer 28.05 kJ


## 6. Tabelas de propriedades para várias temperaturas

`get_properties_range(species_id, [T1, T2, ...])` avalia as propriedades em uma
lista de temperaturas e retorna um dicionário indexado por temperatura.
Temperaturas que estejam fora de todos os intervalos válidos são silenciosamente
ignoradas.

In [11]:
temps = [300, 500, 800, 1200, 1800, 2500]
with ThermochemicalCalculator() as calc:
    co2 = calc.get_available_species("CO2", exact_match=True)[0]["id"]
    table = calc.get_properties_range(co2, temps)

print(f"{'T [K]':>7} {'Cp [J/mol/K]':>14} {'S [J/mol/K]':>13} {'H [kJ/mol]':>12}")
for T in temps:
    p = table[T]
    print(f"{T:7.0f} {p['cp']:14.3f} {p['s']:13.3f} {p['h_relative']/1000:12.3f}")

  T [K]   Cp [J/mol/K]   S [J/mol/K]   H [kJ/mol]
    300         37.220       214.016     -393.439
    500         44.624       234.896     -385.201
    800         51.432       257.491     -370.699
   1200         56.347       279.388     -349.030
   1800         59.693       302.964     -314.076
   2500         61.443       322.881     -271.603


Observe como $C_p$ e $S$ do CO₂ crescem fortemente com a temperatura — uma
marca registrada das moléculas poliatômicas, cujos modos vibracionais se tornam
cada vez mais excitados. O notebook 03 explora esse comportamento graficamente.

## 7. Tratamento de erros

`pyglenn` lança exceções específicas em vez de retornar `None`:

* **`TemperatureOutOfRangeError`** — a temperatura solicitada está fora da
  faixa válida da espécie;
* **`DatabaseNotConnectedError`** — um método é chamado antes da conexão;
* **`SpeciesNotFoundError`** — um ID de espécie inválido é fornecido.

Todas herdam de **`ThermoCalcError`**, então você pode capturá-las individualmente
ou juntas com `try/except`.

In [12]:
from pyglenn import ThermoCalcError, TemperatureOutOfRangeError, DatabaseNotConnectedError

print("--- Temperatura fora da faixa ---")
with ThermochemicalCalculator() as calc:
    o2 = calc.get_available_species("O2", exact_match=True)[0]["id"]
    # O2 gasoso é válido de 200 K a 6000 K, mas 50 000 K está fora da faixa:
    try:
        props = calc.calculate_properties(o2, 50_000.0)
    except TemperatureOutOfRangeError as e:
        print(f"Capturado: {e}")

print("\n--- Chamar antes de conectar ---")
lonely = ThermochemicalCalculator()
try:
    props = lonely.calculate_properties(o2, 300.0)
except DatabaseNotConnectedError as e:
    print(f"Capturado: {e}")

--- Temperatura fora da faixa ---
Capturado: Temperature 50000.0 K out of valid range for species 'O2'. Available intervals: [(200.0, 1000.0), (1000.0, 6000.0), (6000.0, 20000.0)]

--- Chamar antes de conectar ---
Capturado: calculate_properties called without database connection


In [13]:
from pyglenn import (
    ThermoCalcError,
    DatabaseNotConnectedError,
    SpeciesNotFoundError,
    TemperatureOutOfRangeError,
)

# Todas herdam de ThermoCalcError, então você pode capturar toda a família de uma vez.
for exc in (DatabaseNotConnectedError, SpeciesNotFoundError, TemperatureOutOfRangeError):
    print(f"{exc.__name__:28s} é um ThermoCalcError? {issubclass(exc, ThermoCalcError)}")

DatabaseNotConnectedError    é um ThermoCalcError? True
SpeciesNotFoundError         é um ThermoCalcError? True
TemperatureOutOfRangeError   é um ThermoCalcError? True


## Resumo

Agora você sabe como:

- conectar ao banco de dados empacotado (preferencialmente via gerenciador de
  contexto);
- inspecioná-lo com `get_statistics()`;
- buscar espécies com `get_available_species()` e resolver nomes exatos;
- calcular $C_p^\circ$, $S^\circ$ e $H^\circ$ com `calculate_properties()`;
- interpretar `h_relative` como a entalpia padronizada (que contém
  $\Delta_f H^\circ$);
- construir tabelas com `get_properties_range()` e diferenças de entalpia com
  `calculate_enthalpy_change()`.

**A seguir:** o notebook 02 abre a caixa-preta dos polinômios da NASA e mostra
como esses números são calculados a partir dos coeficientes brutos.